# Reaction Order and Labels

When a meta-reaction has multiple reactants of the same meta-species, MobsPy must decide how to map reactant states to product states. By default it uses **round-robin** assignment: the first product inherits the state of the first reactant, the second product inherits the second reactant's state, and so on, cycling back to the first reactant if there are more products than reactants.

You can override this default with **labels**.

In [ ]:
from mobspy import *

## Round-Robin Assignment

Consider a competition reaction where two individuals of the same species fight. The species has two color states, `red` and `blue`. With round-robin order, MobsPy assigns the first product the state of the first reactant.

In [ ]:
S = BaseSpecies()
S.red, S.blue

# The surviving individual keeps the state of the first reactant
S + S >> S[1]

sim = Simulation(S)
print(sim.compile())

## Labels

The `.label()` method lets you explicitly control which reactant maps to which product. Products inherit the state of the reactant with the same label, regardless of position.

In the next reaction, we want the *second* reactant to be the survivor. We label it and place the same label on the product.

In [ ]:
S2 = BaseSpecies()
S2.red, S2.blue

# The second reactant survives (label matching)
S2 + S2.label(1) >> S2.label(1)[2]

sim2 = Simulation(S2)
print(sim2.compile())

Compare the two compilations above. In the first (round-robin), `S.red + S.blue >> S.red` because the product takes the first reactant's state. In the second (labeled), `S2.red + S2.blue >> S2.blue` because the product is labeled to match the second reactant.

## Labels in Reproduction Reactions

Labels are essential for reproduction reactions where a parent produces offspring with the same state. Without labels, round-robin naturally handles the simple case of `A >> 2*A` (both products get the single reactant's state). But when two different meta-species interact, labels give you precise control.

In [ ]:
Bacteria = BaseSpecies()
Bacteria.healthy, Bacteria.sick

# A healthy bacterium eats a sick one and reproduces.
# The offspring should be healthy (same as the first reactant).
# The sick one is consumed.
Bacteria.healthy.label("a") + Bacteria.sick >> 2 * Bacteria.label("a")[0.5]

sim3 = Simulation(Bacteria)
print(sim3.compile())

## Programmatic Characteristics with `.c()`

When looping over characteristics in Python, you must use `.c(variable)` instead of dot notation. Python resolves `.name` as the literal string `name`, not the value of a variable called `name`.

In [ ]:
# WRONG: this adds characteristics literally named 'color', not 'red'/'blue'/'green'
Wrong = BaseSpecies()
colors = ["red", "blue", "green"]
for color in colors:
    Wrong.color >> Zero[1]  # adds 'color' three times, not 'red', 'blue', 'green'

sim_wrong = Simulation(Wrong)
print("=== WRONG ===")
print(sim_wrong.compile())

In [ ]:
# RIGHT: .c() evaluates the variable
Right = BaseSpecies()
colors = ["red", "blue", "green"]
for color in colors:
    Right.c(color) >> Zero[1]

sim_right = Simulation(Right)
print("=== RIGHT ===")
print(sim_right.compile())

The wrong version creates a single characteristic called `color` with one reaction. The right version creates three characteristics (`red`, `blue`, `green`) with one reaction each.